#Criação da tabela Silver - GOALS

## Objetivos
- Criação de tabela que capture dados da bronze sem ocorrência de duplicatas e apenas de partidas encerradas
- Explode do array "goalscorer" que contém as informações de gols marcados
- Uso das informações do array para a criação de novas colunas

In [0]:
from pyspark.sql.functions import *

In [0]:
# coletando o máximo registro e transformando em uma váriavel para usar como filtro
max_dh_bronze = (
    spark.read.table(
        "api_football.bronze.matches_raw"
    )  # leitura da tabela bronze pra coleta da última data de atualização
    .agg(
        max("dh_processing_br").alias("max_dh_bronze")
    )  # variável que captura a data mais recente de atualização (maior)
    .collect()[0]["max_dh_bronze"]  # transforma de dataframe para variável
)

# leitura da tabela de comparação, nesse caso a goals
df_silver = spark.read.table("api_football.silver.goals")

# join para pegar os dados que não estão na silver
df_matches_raw = (
    spark.read.table("api_football.bronze.matches_raw")
    .alias("mr")
    .join(
        df_silver.alias("s"), col("s.match_id") == col("mr.match_id"), "leftanti"
    )  # left anti pega os dados das match_id que estão na bronze, mas não estão na silver
    .filter(
        col("mr.dh_processing_br") == max_dh_bronze
    )  # filtra somente os dados que tem a data de atualização mais recente
    .filter(
        col("mr.match_status") == "Finished"
    )  # filtra somente as partidas finalizadas
)

In [0]:
#df_matches_raw.printSchema()

In [0]:
#seleção de colunas do dataframe matches_raw que vão compor o dataframe goals_exploded
df_goals_exploded = (
    df_matches_raw
    .select(
        "match_id",
        "match_hometeam_id",
        "match_hometeam_name",
        "match_awayteam_id",
        "match_awayteam_name",
        "dh_processing_br",
        explode("goalscorer").alias("goal") #explode do array
    )
)

#display(df_goals_exploded)

In [0]:
# seleção de colunas do dataframe goals_exploded que vão compor o dataframe goals
df_goals = df_goals_exploded.select(
    col("match_id"),
    # se o campo de home_scorer for diferente de vazio, ele vai retornar o campo home_scorer, caso contrário ele vai retornar o campo away_scorer e criar a coluna player_name
    when(col("goal.home_scorer") != "", col("goal.home_scorer"))
    .otherwise(col("goal.away_scorer"))
    .alias("player_name"),
    # se o campo de home_assist for diferente de vazio, ele vai retornar o campo home_assist, caso contrário ele vai retornar o campo away_assist e criar a coluna assist_name
    when(col("goal.home_assist") != "", col("goal.home_assist"))
    .otherwise(col("goal.away_assist"))
    .alias("assist_name"),
    # captura o campo time e cria a coluna score_time
    col("goal.time").alias("score_time"),
    # captura o campo score e cria a coluna score
    col("goal.score").alias("score"),
    # se o campo home_scorer for diferente de vazio, ele vai retornar o campo match_hometeam_id, caso contrário ele vai retornar o campo match_awayteam_id e criar a coluna team_id, com o id so time que fez o gol
    when(col("goal.home_scorer") != "", col("match_hometeam_id"))
    .otherwise(col("match_awayteam_id"))
    .alias("team_id"),
    # se o campo home_scorer for diferente de vazio, ele vai retornar o campo match_hometeam_name, caso contrário ele vai retornar o campo match_awayteam_name e criar a coluna team_name, com o nome do time que fez o gol
    when(col("goal.home_scorer") != "", col("match_hometeam_name"))
    .otherwise(col("match_awayteam_name"))
    .alias("team_name"),
    # se o campo home_scorer for diferente de vazio, ele vai retornar o campo True, caso contrário ele vai retornar o campo False e criar a coluna is_home_team, com o valor True ou False se o time que fez o gol é o time da casa ou não
    when(col("goal.home_scorer") != "", lit(True))
    .otherwise(lit(False))
    .alias("is_home_team"),
    # coluna que captura a data de atualização da bronze
    col("dh_processing_br").alias("dh_processing_bronze_br"),
    # coluna que captura a data de atualização do dataframe atual
    expr("current_timestamp()-interval 3 hours").alias("dh_processing_br"),
)

# display(df_goals)

In [0]:
df_goals.write.mode("append").saveAsTable("api_football.silver.goals")

In [0]:
%sql
select
  *
from
  api_football.silver.goals